In [3]:
from dotenv import dotenv_values
from openai import DefaultHttpxClient, OpenAI

# dotenv_values 把 .env 里的“名称=值”读成字典。相对路径从内核的工作目录开始查找；
# 在 Jupyter 中打开本 notebook 时，工作目录通常就是它所在的文件夹。
config = dotenv_values(".env")

# API Key 用于身份验证，代理地址决定连接从哪里发出。
# 这里只取需要的两个值。打印整个 config 会把密钥一起写进 notebook 输出。
api_key = config.get("OPENAI_API_KEY")
proxy = config.get("OPENAI_PROXY")
if not api_key or not proxy:
    raise ValueError("请检查同目录的 .env 是否填写了 OPENAI_API_KEY 和 OPENAI_PROXY。")

In [4]:
# OpenAI 是官方 SDK 的客户端。SDK 会把 api_key 放进请求的认证头。
# base_url 指定 OpenAI 官方 API 地址；代理只负责转发连接。
# proxy 显式使用 .env 中的代理，trust_env=False 避免系统代理环境变量影响本次测试。
# with 语句在代码块结束时自动关闭客户端，释放网络连接。
with OpenAI(
    api_key=api_key,
    base_url="https://api.openai.com/v1",
    http_client=DefaultHttpxClient(proxy=proxy, trust_env=False),
    timeout=30.0,  # 网络连接或读取长时间无响应时触发超时，单位为秒。
) as client:
    # 执行 create() 才会发起模型调用。重新运行这个单元格会产生新的请求和用量。
    response = client.responses.create(
        model="gpt-6-luna",
        input="Reply with exactly OK.",  # 提示词：要求只回复 OK，方便判断测试结果。
        reasoning={"effort": "none"},  # 本题只需回复固定文本，选择模型支持的无推理模式。
        max_output_tokens=16,  # 限制生成的 token 数量；实际生成量可以更少。
    )

In [5]:
# response 是 SDK 返回的 Python 对象。model 是服务端报告的模型名，
# output_text 会把本次响应中的文本片段拼起来，可直接用于显示。
# 单独重跑这个单元格只读取上一次的结果，不会再次调用 API。
print("模型：", response.model)
print("回复：", response.output_text)

# token 是模型处理文本时使用的单位，一个中文字或英文单词都可能拆成多个 token。
# usage 记录本次请求实际使用的输入和输出 token 数，可结合模型单价估算费用。
if response.usage is not None:
    print(f"用量：输入 {response.usage.input_tokens} tokens，输出 {response.usage.output_tokens} tokens")

模型： gpt-6-luna
回复： OK
用量：输入 11 tokens，输出 5 tokens
